In [ ]:
from pathlib import Path

# raíz del proyecto
ROOT = Path().resolve().parent

# carpetas de datos
DATA_RAW = ROOT / "data" / "raw"
DATA_INTERIM = ROOT / "data" / "interim"
DATA_PROCESSED = ROOT / "data" / "processed"

REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
OUTPUTS = REPORTS / "outputs"

In [7]:
# Carga y limpieza estructural
import pandas as pd
import numpy as np

gyms_raw = pd.read_excel(DATA_RAW / "Gimnasios de albacete.xlsx")

# 1) borrar filas totalmente vacías
gyms = gyms_raw.dropna(how="all")

# 2) borrar columnas totalmente vacías (típicas Unnamed)
gyms = gyms.dropna(axis=1, how="all")

# 3) si aún quedan más de 6, nos quedamos con las primeras 6
gyms = gyms.iloc[:, :6].copy()

# 4) ahora sí: renombrar
gyms.columns = ["nombre", "direccion", "rating", "reviews", "categoria", "horario"]

# 5) borrar filas sin nombre
gyms = gyms[gyms["nombre"].notna()].reset_index(drop=True)

gyms.head()

,nombre,direccion,rating,reviews,categoria,horario
0,2026-03-03 00:00:00,Datos recojidos manualmente por medio de googl...,NaN,NaN,NaN,NaN
1,NOMBRE,DIRECCION,CLASIFICACION,RESEÑAS,CATEGORIA,ABIRTO/24
2,Fitness Park,"C. Alcalde Conangla, s/n, 02006 Albacete",2026-09-04 00:00:00,200,Gimnasio,06:00 a 1:00
3,Synergym Albacete,"P.º la Cuba, 12, 02005 Albacete",2026-05-04 00:00:00,300,Gimnasio,6:30 a 22:30
4,Synergym Universidad,"C. Hellín, 12, 02006 Albacete",2026-08-04 00:00:00,300,Gimnasio,6:00 a 23:00


In [ ]:
# Limpieza de rating y reviews
#  Rating a float
gyms["rating"] = pd.to_numeric(gyms["rating"], errors="coerce")

# Reviews a numérico (quitar texto si hay)
gyms["reviews"] = gyms["reviews"].astype(str).str.replace(",", "")
gyms["reviews"] = pd.to_numeric(gyms["reviews"], errors="coerce")

In [ ]:
# Asegurarnos de que la columna sea string
gyms["categoria"] = gyms["categoria"].astype(str).str.lower().str.strip()

def clean_category(x):
    if x in ["nan", "none", ""]:
        return "otros"
    if "cross" in x:
        return "crossfit"
    if "pilates" in x:
        return "pilates"
    if "yoga" in x:
        return "yoga"
    if "boxeo" in x:
        return "boxeo"
    if "gimnasio" in x:
        return "gimnasio_general"
    return "otros"

gyms["categoria_limpia"] = gyms["categoria"].apply(clean_category)

In [ ]:
# 4️⃣ Métricas descriptivas iniciales
print("Total gimnasios:", len(gyms))
print("Rating medio:", gyms["rating"].mean())
print("Reviews medias:", gyms["reviews"].mean())

gyms["categoria_limpia"].value_counts()

Total gimnasios: 98
Rating medio: 4.9714285714285715
Reviews medias: 118.33333333333333


categoria_limpia
gimnasio_general    60
otros               14
pilates             13
boxeo                7
yoga                 4
Name: count, dtype: int64

### Dataset por municipio (modelo replicable)

In [ ]:

import requests
import pandas as pd
from io import BytesIO

url_xlsx = "https://www.ine.es/jaxiT3/files/t/xlsx/2855.xlsx"

# Descargar (saltando verificación SSL por el problema de tu PC)
r = requests.get(url_xlsx, verify=False, timeout=60)
r.raise_for_status()

# Leer Excel desde memoria
poblacion_raw = pd.read_excel(BytesIO(r.content))

print(poblacion_raw.shape)
print(poblacion_raw.columns)
poblacion_raw.head()

c:\Users\juanc\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.ine.es'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


(103, 91)
Index(['Cifras oficiales de población de los municipios españoles en aplicación de la Ley de Bases del Régimen Local (Art. 17)',
       'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10',
       'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14',
       'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18',
       'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22',
       'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26',
       'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30',
       'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34',
       'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38',
       'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42',
       'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46',
       'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50',
       'Unnamed: 51', 'Unnamed: 52',

c:\Users\juanc\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Cifras oficiales de población de los municipios españoles en aplicación de la Ley de Bases del Régimen Local (Art. 17),Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 81,Unnamed: 82,Unnamed: 83,Unnamed: 84,Unnamed: 85,Unnamed: 86,Unnamed: 87,Unnamed: 88,Unnamed: 89,Unnamed: 90
0,Detalle municipal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Albacete: Población por municipios y sexo.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Unidades: Personas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd

pob_raw = pd.read_excel("poblacion_ine_2855.xlsx")

pob_raw.shape

c:\Users\juanc\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(103, 91)

In [ ]:
import pandas as pd

pob_raw = pd.read_excel("poblacion_ine_2855.xlsx", header=5)
pob_raw.head()

c:\Users\juanc\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 81,Unnamed: 82,Unnamed: 83,Unnamed: 84,Unnamed: 85,Unnamed: 86,Unnamed: 87,Unnamed: 88,Unnamed: 89,Unnamed: 90
0,,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,,2025,2024.0,2023.0,2022.0,2021.0,2020.0,2019.0,2018.0,2017.0,...,2005.0,2004.0,2003.0,2002.0,2001.0,2000,1999,1998,1997,1996
2,02001 Abengibre,753,759.0,760.0,739.0,748.0,761.0,790.0,757.0,748.0,...,494.0,494.0,479.0,494.0,500.0,497,503,508,,513
3,02002 Alatoz,504,497.0,491.0,496.0,496.0,506.0,519.0,529.0,555.0,...,280.0,288.0,297.0,299.0,311.0,316,328,324,,326
4,02003 Albacete,175400,174137.0,173206.0,172357.0,172722.0,174336.0,173329.0,173050.0,172816.0,...,81102.0,79605.0,78891.0,77426.0,76229.0,76383,75388,74328,,73565


### Scraping

In [ ]:
!pip install python-dotenv


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
###############################################################
# SECCIÓN 1 — LIBRERÍAS
###############################################################

import time
import re
import requests
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

from dotenv import load_dotenv
import os


###############################################################
# SECCIÓN 2 — CONFIGURACIÓN DEL PROYECTO
###############################################################

# cargar variables del archivo .env
load_dotenv()

API_KEY = os.getenv("GOOGLE_API_KEY")
if not API_KEY:
    raise ValueError("No se encontró GOOGLE_API_KEY en el archivo .env")

INPUT_CSV  = "dataset_municipios_albacete_places.csv"
OUTPUT_CSV = "dataset_municipios_albacete_places_enriquecido_v3.csv"
RAW_PLACES_CSV = "places_all_results_v4.csv"

GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"
PLACES_URL  = "https://places.googleapis.com/v1/places:searchText"

HEADERS = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.types,places.location,places.formattedAddress"
}


###############################################################
# SECCIÓN 3 — FILTROS DE NOMBRES (ANTI FALSOS POSITIVOS)
###############################################################

# Cosas que NO queremos contar como oferta fitness aunque Google las devuelva
EXCLUDE_NAME = re.compile(r"""
hotel|hostal|resort|spa|balneario|camping|
kart|karting|paintball|
piscina|campo|futbol|fútbol|estadio|pista|
front[oó]n|fronton|
parque|
ayuntamiento|
centro\s+social|
polideportivo|pabell[oó]n|complejo|
multiaventura|aventura|piragua|piraguas|despedida|rafting|kayak
""", re.IGNORECASE | re.VERBOSE)

# Palabras que SÍ suelen indicar oferta fitness real (cuando los types no ayudan)
INCLUDE_KEYWORDS = re.compile(r"""
gimnasio|gym|fitness|fit|
crossfit|cross|box|
training|entrenamiento|personal\s*trainer|
pilates|yoga|
centro\s+deportivo|club\s+deportivo|
body|performance|wellness
""", re.IGNORECASE | re.VERBOSE)

# Types ampliados (depende de lo que devuelva Places, esto cubre más casos)
ALLOW_TYPES = {
    "gym",
    "fitness_center",
    "health_club",
    "sports_club",
    "sports_complex",
}

###############################################################
# SECCIÓN 4 — CÁLCULO DE DISTANCIA (HAVERSINE)
###############################################################

def haversine(lat1, lon1, lat2, lon2):

    R = 6371000

    lat1, lon1, lat2, lon2 = map(radians,[lat1,lon1,lat2,lon2])

    dlat = lat2-lat1
    dlon = lon2-lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2*atan2(sqrt(a),sqrt(1-a))

    return R*c


###############################################################
# SECCIÓN 5 — GEOCODING DE MUNICIPIOS
###############################################################

def geocode_municipio(municipio):

    address = f"{municipio}, Albacete, Spain"

    params = {
        "address": address,
        "key": API_KEY
    }

    r = requests.get(GEOCODE_URL, params=params)

    data = r.json()

    if data.get("status") != "OK":
        return None, None

    result = data["results"][0]

    loc = result["geometry"]["location"]

    lat = loc["lat"]
    lon = loc["lng"]

    return lat, lon

###############################################################
# SECCIÓN 6 — RADIO DE BÚSQUEDA
###############################################################

def choose_radius(pop):

    if pop < 1000:
        return 4000

    if pop < 5000:
        return 6000

    if pop < 20000:
        return 10000

    return 18000


###############################################################
# SECCIÓN 7 — BÚSQUEDA DE OFERTA FITNESS (MULTI-QUERY)
###############################################################

def places_search_fitness(municipio, lat, lon, radius):
    
    queries = [
        f"gimnasio en {municipio}",
        f"fitness en {municipio}",
        f"centro deportivo en {municipio}",
        f"pilates en {municipio}",
        f"yoga en {municipio}",
        f"crossfit en {municipio}",
        f"entrenamiento personal en {municipio}",
    ]
    
    unique = {}      # pid -> dict con info
    raw_rows = []    # para guardar en CSV crudo
    
    for q in queries:
        payload = {
            "textQuery": q,
            "languageCode": "es",
            "regionCode": "ES",
            "locationBias": {
                "circle": {
                    "center": {"latitude": lat, "longitude": lon},
                    "radius": radius
                }
            }
        }

        resp = requests.post(PLACES_URL, headers=HEADERS, json=payload)
        data = resp.json()

        for p in data.get("places", []):
            pid = p.get("id")
            name = p.get("displayName", {}).get("text", "")
            types = set(p.get("types", []))
            loc = p.get("location", {})
            addr = p.get("formattedAddress", "")

            if not pid or not name or not loc:
                continue

            lat2 = loc.get("latitude")
            lon2 = loc.get("longitude")
            if lat2 is None or lon2 is None:
                continue

            # filtro distancia real
            dist_m = haversine(lat, lon, lat2, lon2)
            if dist_m > radius:
                continue

            # excluir falsos positivos por nombre
            if EXCLUDE_NAME.search(name):
                continue

            # criterio de aceptación:
            # - por type
            # - o por keywords en el nombre (porque muchos no vienen con types "gym")
            allowed_by_type = any(t in ALLOW_TYPES for t in types)
            allowed_by_name = bool(INCLUDE_KEYWORDS.search(name))

            if not (allowed_by_type or allowed_by_name):
                continue

            # guardar raw (para auditoría)
            raw_rows.append({
                "municipio": municipio,
                "query": q,
                "place_id": pid,
                "name": name,
                "types": ";".join(sorted(types)),
                "lat": lat2,
                "lon": lon2,
                "distance_m": dist_m,
                "formattedAddress": addr
            })

            # dedupe por place_id (nos quedamos con 1)
            if pid not in unique:
                unique[pid] = {
                    "name": name,
                    "types": types
                }

        # pausa chica para no ir a lo loco (y respetar cuota)
        time.sleep(0.15)

    names = sorted([v["name"] for v in unique.values()])

    return len(unique), "; ".join(names[:8]), raw_rows

###############################################################
# SECCIÓN 8 — PIPELINE PRINCIPAL
###############################################################

df = pd.read_csv(INPUT_CSV)

rows = []
all_raw = []  # guardamos todos los resultados crudos

for i, row in df.iterrows():

    municipio = row["municipio"]
    pop       = row["poblacion_2025"]

    lat, lon = geocode_municipio(municipio)

    if lat is None:
        rows.append({**row, "gyms_google_new": None})
        continue

    radius = choose_radius(pop)

    n, names, raw_rows = places_search_fitness(municipio, lat, lon, radius)
    all_raw.extend(raw_rows)

    rows.append({
        **row,
        "lat": lat,
        "lon": lon,
        "radius_m": radius,
        "gyms_google_new": n,
        "gyms_sample_names": names
    })

    if i % 10 == 0:
        print(i, "municipios procesados")

    time.sleep(0.2)  # pausa entre municipios

###############################################################
# SECCIÓN 9 — RESULTADOS
###############################################################

out = pd.DataFrame(rows)
out["fitness_x10k_new"] = (out["gyms_google_new"] / out["poblacion_2025"]) * 10000
out.to_csv(OUTPUT_CSV, index=False)

# Guardar resultados crudos (MUY útil para recalcular filtros sin API)
raw_df = pd.DataFrame(all_raw)
raw_df.to_csv(RAW_PLACES_CSV, index=False)

print("Guardado:", OUTPUT_CSV)
print("Guardado:", RAW_PLACES_CSV)

print("\nTOP sospechosos\n")
print(
    out.sort_values("fitness_x10k_new", ascending=False)[
        ["municipio", "poblacion_2025", "gyms_google_new", "fitness_x10k_new"]
    ].head(15)
)

0 municipios procesados
10 municipios procesados
20 municipios procesados
30 municipios procesados
40 municipios procesados
50 municipios procesados
60 municipios procesados
70 municipios procesados
80 municipios procesados
Guardado: dataset_municipios_albacete_places_enriquecido_v3.csv
Guardado: places_all_results_v4.csv

TOP sospechosos

                   municipio  poblacion_2025  gyms_google_new  \
82                 Villatoya             114                1   
6           Alcalá del Júcar            1130                8   
30                 Fuensanta             282                1   
39              Hoya-Gonzalo             582                2   
41                     Letur             911                3   
37               Herrera, La             308                1   
67                    Riópar            1344                4   
19                  Carcelén             483                1   
50  Montealegre del Castillo            2051                4   
22      

1️⃣ Qué significa realmente tu tabla

La métrica que estás viendo es:

𝑓𝑖𝑡𝑛𝑒𝑠𝑠_𝑥10𝑘 = 𝑔𝑖𝑚𝑛𝑎𝑠𝑖𝑜𝑠  
-- -- -- -- -- -- 𝑝𝑜𝑏𝑙𝑎𝑐𝑖𝑜𝑛     × 10 000

O sea:
alto valor → mucha oferta para poca gente
bajo valor → poca oferta para mucha gente
Para abrir gimnasio, nos interesan los valores bajos, no los altos.

In [ ]:
oportunidad = out[
    (out["poblacion_2025"] > 2000) &
    (out["gyms_google_new"] <= 1)
].sort_values("poblacion_2025", ascending=False)

print(oportunidad[
    ["municipio","poblacion_2025","gyms_google_new","fitness_x10k_new"]
].head(20))

                     municipio  poblacion_2025  gyms_google_new  fitness_x10k_new
23                Casas-Ibáñez            4616                1          2.166378
26  Chinchilla de Monte-Aragón            4614                1          2.167317
29          Elche de la Sierra            3603                1          2.775465
52                      Munera            3351                0          0.000000
61                 Pozo Cañada            2704                0          0.000000
34                  Gineta, La            2656                0          0.000000
86                       Yeste            2471                1          4.046945
31                Fuente-Álamo            2422                1          4.128819
11                    Balazote            2356                1          4.244482
9                       Alpera            2278                0          0.000000
50    Montealegre del Castillo            2051                0          0.000000


In [ ]:
# Crear un indicador de mercado local
out["market_potential"] = out["poblacion_2025"] / (out["gyms_google_new"] + 1)

ranking = out.sort_values("market_potential", ascending=False)

print(
ranking[
["municipio","poblacion_2025","gyms_google_new","market_potential"]
].head(15)
)

                     municipio  poblacion_2025  gyms_google_new  market_potential
2                     Albacete          175400               19       8770.000000
36                      Hellín           30836                4       6167.200000
52                      Munera            3351                0       3351.000000
61                 Pozo Cañada            2704                0       2704.000000
34                  Gineta, La            2656                0       2656.000000
24                     Caudete           10439                3       2609.750000
69                    Roda, La           15643                5       2607.166667
81               Villarrobledo           25400                9       2540.000000
23                Casas-Ibáñez            4616                1       2308.000000
26  Chinchilla de Monte-Aragón            4614                1       2307.000000
9                       Alpera            2278                0       2278.000000
50    Montealegr

In [ ]:
# Filtrar solo pueblos interesantes
ranking2 = ranking[
ranking["poblacion_2025"] > 2000
]

print(
ranking2[
["municipio","poblacion_2025","gyms_google_new","market_potential"]
].head(15)
)

                     municipio  poblacion_2025  gyms_google_new  market_potential
2                     Albacete          175400               19       8770.000000
36                      Hellín           30836                4       6167.200000
52                      Munera            3351                0       3351.000000
61                 Pozo Cañada            2704                0       2704.000000
34                  Gineta, La            2656                0       2656.000000
24                     Caudete           10439                3       2609.750000
69                    Roda, La           15643                5       2607.166667
81               Villarrobledo           25400                9       2540.000000
23                Casas-Ibáñez            4616                1       2308.000000
26  Chinchilla de Monte-Aragón            4614                1       2307.000000
9                       Alpera            2278                0       2278.000000
50    Montealegr

4️⃣ Los que realmente me parecen interesantes

🥇 Munera
- 3351 habitantes
- solo gimnasio municipal
- lejos de grandes ciudades
- 👉 Muy buen candidato

🥈 Pozo Cañada
- 2704 habitantes
- 0 gimnasios
- tamaño perfecto para gimnasio pequeño
- 👉 muy interesante

🥉 Casas-Ibáñez
- 4616 habitantes
- 1 gimnasio

4️⃣ Elche de la Sierra
- 3600 habitantes
- solo 1 gimnasio
- bastante aislado

5️⃣ Alpera

Esto es interesante porque: 4500 personas para 1 gimnasio - probablemente hay hueco para otro